In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import functools
import numpy as np
import matplotlib.pyplot as plt
import os
from torchvision import transforms
from math import pi
import SimpleITK as sitk
import plotly
import plotly.express as px
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import vtk
from vtk.util.numpy_support import vtk_to_numpy
import pickle
from monai.transforms import (RandSimulateLowResolution)
import monai 
import sys

In [ ]:
sys.path.append('/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl/')

In [ ]:
from nets.cut import Cut
from nets.lotus import UltrasoundRenderingCutTarget

In [ ]:
df = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/acoustic_params_v3.csv')

In [ ]:
model_cut = Cut.load_from_checkpoint('/mnt/raid/C1_ML_Analysis/train_output/ultra-sim/rendering_cut/v0.12/epoch=24-val_loss=3.43.ckpt')
model_cut = model_cut.eval()
model_cut = model_cut.cuda()

In [ ]:
df_cut = df.copy()
df_cut['acoustic_impedance_dict'] = model_cut.USR.acoustic_impedance_dict.detach().cpu().numpy()
df_cut['attenuation_dict'] = model_cut.USR.attenuation_dict.detach().cpu().numpy()
df_cut['mu_0_dict'] = model_cut.USR.mu_0_dict.detach().cpu().numpy()
df_cut['mu_1_dict'] = model_cut.USR.mu_1_dict.detach().cpu().numpy()
df_cut['sigma_0_dict'] = model_cut.USR.sigma_0_dict.detach().cpu().numpy()

In [ ]:
fig = px.violin(df_cut, x='group', y="acoustic_impedance_dict")
fig.show()

In [ ]:
fig = px.violin(df_cut, x='group', y="attenuation_dict")
fig.show()

In [ ]:
fig = px.violin(df_cut, x='group', y="mu_0_dict")
fig.show()

In [ ]:
fig = px.violin(df_cut, x='group', y="mu_1_dict")
fig.show()

In [ ]:
fig = px.violin(df_cut, x='group', y="sigma_0_dict")
fig.show()

In [ ]:
df_v4 = pd.read_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/acoustic_params_v4.csv')

In [ ]:
df_v4['acoustic_impedance_dict'] = df_cut['acoustic_impedance_dict']
df_v4['attenuation_dict'] = df_cut['attenuation_dict']
df_v4['mu_0_dict'] = df_cut['mu_0_dict']
df_v4['mu_1_dict'] = df_cut['mu_1_dict']
df_v4['sigma_0_dict'] = df_cut['sigma_0_dict']

df_v4['name'] = df_v4['img_fn'].apply(lambda x: str(x).replace('.nrrd', '').split('/')[-1])

In [ ]:
# Grouping by 'group' and calculating the mean of 'mean'
group_means = df_v4.groupby(['label_11']).agg({
    'acoustic_impedance_dict': 'mean',
    'attenuation_dict': 'mean',
    'mu_0_dict': 'mean',
    'mu_1_dict': 'mean',
    'sigma_0_dict': 'mean',
    'mean': 'mean',  # If you still need the mean of the 'mean' column
    'std': 'mean'    # Similarly, if you need the mean of 'std'
}).reset_index()

print(group_means)
group_means.to_csv('/mnt/raid/C1_ML_Analysis/source/blender/simulated_data_export/acoustic_params_v4_group_means.csv', index=False)

In [ ]:
from monai.transforms import (        
    RandSpatialCrop,    
)

RandSpatialCrop((128, 128, 128), random_size=False)(torch.rand(1, 256, 256, 256)).shape